# Maven 电商 Billing Page A/B Test

本 Notebook 只分析主实验结果：Billing-to-Purchase Conversion Rate 和 Revenue per Billing Session。

## 1. 导入统计分析包

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. 读取实验样本

CSV 由 SQL 提取，一行代表一个 Billing 实验 Session。

In [2]:
file_path = '../output/tables/ab_experiment_sample.csv'
df = pd.read_csv(file_path, parse_dates=['billing_exposure_at'])
df.head()

,website_session_id,user_id,billing_version,billing_pageview_id,billing_exposure_at,device_type,is_repeat_session,utm_source,utm_campaign,purchased,order_count,revenue
0,25325,23266,treatment,53550,2012-09-10 00:13:05,desktop,0,gsearch,nonbrand,1,1,49.9900
1,25343,23284,control,53583,2012-09-10 05:39:09,desktop,0,gsearch,nonbrand,1,1,49.9900
2,25353,23291,control,53610,2012-09-10 07:53:37,desktop,0,gsearch,nonbrand,1,1,49.9900
3,25358,23064,treatment,53628,2012-09-10 08:36:13,desktop,1,gsearch,brand,1,1,49.9900
4,25368,23305,control,53662,2012-09-10 09:26:12,desktop,0,gsearch,nonbrand,1,1,49.9900


## 3. 基础数据确认

检查样本行数、Session 唯一性、实验版本、购买标记和关键缺失。

In [3]:
quality_check = pd.Series({
    '数据行数': len(df),
    'website_session_id 唯一': df['website_session_id'].is_unique,
    'purchased 只有 0/1': set(df['purchased'].dropna().unique()).issubset({0, 1}),
    'purchased 缺失数': df['purchased'].isna().sum(),
    'revenue 缺失数': df['revenue'].isna().sum()
})
display(quality_check.to_frame('结果'))

version_distribution = df['billing_version'].value_counts().rename_axis('billing_version').to_frame('sessions')
purchased_distribution = df['purchased'].value_counts(dropna=False).sort_index().rename_axis('purchased').to_frame('sessions')
display(version_distribution)
display(purchased_distribution)

,结果
数据行数,1311
website_session_id 唯一,True
purchased 只有 0/1,True
purchased 缺失数,0
revenue 缺失数,0


,sessions
billing_version,
control,657
treatment,654


,sessions
purchased,
0,601
1,710


## 4. Control 与 Treatment 描述性指标

Purchase CR = 购买 Session 数 / Billing 实验 Session 数；Revenue per Billing Session = Revenue / Sessions。

In [4]:
group_summary = (
    df.groupby('billing_version')
      .agg(
          sessions=('website_session_id', 'count'),
          purchasing_sessions=('purchased', 'sum'),
          revenue=('revenue', 'sum')
      )
      .reindex(['control', 'treatment'])
)
group_summary['purchase_cr'] = group_summary['purchasing_sessions'] / group_summary['sessions']
group_summary['revenue_per_billing_session'] = group_summary['revenue'] / group_summary['sessions']
group_summary

,sessions,purchasing_sessions,revenue,purchase_cr,revenue_per_billing_session
billing_version,,,,,
control,657,300,"14,997.0000",0.4566,22.8265
treatment,654,410,"20,495.9000",0.6269,31.3393


## 5. Purchase CR 提升幅度

Absolute Lift 为 Treatment CR 减去 Control CR；Relative Lift 以 Control CR 为基准。

In [5]:
p_control = group_summary.loc['control', 'purchase_cr']
p_treatment = group_summary.loc['treatment', 'purchase_cr']

absolute_lift = p_treatment - p_control
relative_lift = absolute_lift / p_control

print(f'Absolute Lift: {absolute_lift:.4%} ({absolute_lift * 100:.2f} percentage points)')
print(f'Relative Lift: {relative_lift:.2%}')

Absolute Lift: 17.0290% (17.03 percentage points)
Relative Lift: 37.29%


## 6. Two-Proportion Z-Test

- H0：Control 和 Treatment 的真实购买转化率相同。
- H1：两组真实购买转化率不同。
- 显著性水平：$\alpha = 0.05$。

In [6]:
purchasing_control = int(group_summary.loc['control', 'purchasing_sessions'])
purchasing_treatment = int(group_summary.loc['treatment', 'purchasing_sessions'])
n_control = int(group_summary.loc['control', 'sessions'])
n_treatment = int(group_summary.loc['treatment', 'sessions'])

z_statistic, p_value = proportions_ztest(
    count=[purchasing_treatment, purchasing_control],
    nobs=[n_treatment, n_control],
    alternative='two-sided'
)

print(f'Z statistic: {z_statistic:.4f}')
print(f'p-value: {p_value:.6f}')

Z statistic: 6.1872
p-value: 0.000000


## 7. Purchase CR 差值的 95% Confidence Interval

置信区间针对 Treatment CR - Control CR，使用两独立比例差的 non-pooled standard error。

In [7]:
standard_error = np.sqrt(
    p_control * (1 - p_control) / n_control
    + p_treatment * (1 - p_treatment) / n_treatment
)
ci_lower = absolute_lift - 1.96 * standard_error
ci_upper = absolute_lift + 1.96 * standard_error

print(f'95% CI: [{ci_lower:.4%}, {ci_upper:.4%}]')
print(f'95% CI (percentage points): [{ci_lower * 100:.2f}, {ci_upper * 100:.2f}]')

95% CI: [11.7143%, 22.3438%]
95% CI (percentage points): [11.71, 22.34]


## 8. Sample Ratio Mismatch（SRM）检查

预期分流为 50% Control / 50% Treatment，使用 Chi-square goodness-of-fit test。

In [8]:
observed_sessions = np.array([n_control, n_treatment])
expected_sessions = np.array([len(df) / 2, len(df) / 2])
srm_chi2, srm_p_value = chisquare(f_obs=observed_sessions, f_exp=expected_sessions)

print(f'Actual Control Sessions: {n_control}')
print(f'Actual Treatment Sessions: {n_treatment}')
print(f'SRM p-value: {srm_p_value:.6f}')
if srm_p_value > 0.05:
    print('没有发现明显 Sample Ratio Mismatch 证据。')
else:
    print('发现 Sample Ratio Mismatch 迹象，需要进一步检查实验分流。')

Actual Control Sessions: 657
Actual Treatment Sessions: 654
SRM p-value: 0.933967
没有发现明显 Sample Ratio Mismatch 证据。


## 9. 实验结果汇总

In [9]:
control_rpbs = group_summary.loc['control', 'revenue_per_billing_session']
treatment_rpbs = group_summary.loc['treatment', 'revenue_per_billing_session']

result_table = pd.DataFrame({
    'Metric': ['Sessions', 'Purchase CR', 'Revenue per Billing Session'],
    'Control': [f'{n_control:,}', f'{p_control:.2%}', f'${control_rpbs:,.2f}'],
    'Treatment': [f'{n_treatment:,}', f'{p_treatment:.2%}', f'${treatment_rpbs:,.2f}'],
    'Difference / Lift': [
        f'{n_treatment - n_control:+,}',
        f'{absolute_lift * 100:+.2f} pp / {relative_lift:+.2%}',
        f'${treatment_rpbs - control_rpbs:+,.2f}'
    ]
})
display(result_table)

statistical_results = pd.Series({
    'Absolute Lift': f'{absolute_lift * 100:.2f} percentage points',
    'Relative Lift': f'{relative_lift:.2%}',
    'Z statistic': f'{z_statistic:.4f}',
    'p-value': f'{p_value:.6f}',
    '95% CI': f'[{ci_lower * 100:.2f}, {ci_upper * 100:.2f}] percentage points',
    'SRM p-value': f'{srm_p_value:.6f}'
})
display(statistical_results.to_frame('Result'))

,Metric,Control,Treatment,Difference / Lift
0,Sessions,657,654,-3
1,Purchase CR,45.66%,62.69%,+17.03 pp / +37.29%
2,Revenue per Billing Session,$22.83,$31.34,$+8.51


,Result
Absolute Lift,17.03 percentage points
Relative Lift,37.29%
Z statistic,6.1872
p-value,0.000000
95% CI,"[11.71, 22.34] percentage points"
SRM p-value,0.933967


## 10. 业务结论

以下结论由实际样本计算结果自动生成。Revenue per Billing Session 仅作描述性比较。

In [10]:
if p_value < 0.05 and absolute_lift > 0:
    significance_conclusion = 'Treatment 显著提高了 Purchase CR'
elif p_value < 0.05 and absolute_lift < 0:
    significance_conclusion = 'Treatment 的 Purchase CR 显著低于 Control'
else:
    significance_conclusion = '没有足够证据说明 Treatment 显著改变了 Purchase CR'

revenue_direction = '同步提高' if treatment_rpbs > control_rpbs else '没有同步提高'
srm_conclusion = (
    '没有发现明显 SRM 证据'
    if srm_p_value > 0.05
    else '发现明显 SRM 迹象，需要进一步检查实验分流'
)

print(f'1. {significance_conclusion}（双侧 p-value = {p_value:.4f}）。')
print(f'2. Purchase CR 绝对提升 {absolute_lift * 100:.2f} 个百分点，相对提升 {relative_lift:.2%}。')
print(f'3. Revenue per Billing Session {revenue_direction}：Control ${control_rpbs:.2f}，Treatment ${treatment_rpbs:.2f}。')
print(f'4. {srm_conclusion}（SRM p-value = {srm_p_value:.4f}）。')

1. Treatment 显著提高了 Purchase CR（双侧 p-value = 0.0000）。
2. Purchase CR 绝对提升 17.03 个百分点，相对提升 37.29%。
3. Revenue per Billing Session 同步提高：Control $22.83，Treatment $31.34。
4. 没有发现明显 SRM 证据（SRM p-value = 0.9340）。


## 11. Refund Guardrail Analysis

检查 Treatment 提高购买转化率的同时，是否导致订单退款率明显恶化。Refund Rate 定义为 Refunded Orders / Orders；一个订单只要存在至少一条退款记录，就记为一个 Refunded Order。

### 11.1 关联实验订单与退款记录

实验订单必须属于实验 Session，且订单时间不早于该 Session 的 Billing 页面曝光时间。退款表按 `order_id` 去重后再标记退款订单。

In [11]:
orders_path = '../data/orders.csv'
refunds_path = '../data/order_item_refunds.csv'

orders = pd.read_csv(orders_path, parse_dates=['created_at'])
refunds = pd.read_csv(refunds_path, parse_dates=['created_at'])

experiment_orders = df[[
    'website_session_id', 'billing_version', 'billing_exposure_at'
]].merge(
    orders[['order_id', 'website_session_id', 'created_at']],
    on='website_session_id',
    how='inner'
)

experiment_orders = experiment_orders[
    experiment_orders['created_at'] >= experiment_orders['billing_exposure_at']
].copy()

refunded_order_ids = refunds['order_id'].dropna().drop_duplicates()
experiment_orders['refunded_order'] = (
    experiment_orders['order_id'].isin(refunded_order_ids).astype(int)
)

print(f"实验订单数: {experiment_orders['order_id'].nunique():,}")
print(f"实验订单 ID 是否唯一: {experiment_orders['order_id'].is_unique}")

实验订单数: 710
实验订单 ID 是否唯一: True


### 11.2 退款率与 Two-Proportion Z-Test

分别计算两组订单退款率，并使用双侧 Two-Proportion Z-Test 检查差异，显著性水平为 $\alpha = 0.05$。

In [12]:
refund_summary = (
    experiment_orders.groupby('billing_version')
    .agg(
        orders=('order_id', 'nunique'),
        refunded_orders=('refunded_order', 'sum')
    )
    .reindex(['control', 'treatment'])
)
refund_summary['refund_rate'] = (
    refund_summary['refunded_orders'] / refund_summary['orders']
)

control_orders = int(refund_summary.loc['control', 'orders'])
treatment_orders = int(refund_summary.loc['treatment', 'orders'])
control_refunded = int(refund_summary.loc['control', 'refunded_orders'])
treatment_refunded = int(refund_summary.loc['treatment', 'refunded_orders'])
control_refund_rate = refund_summary.loc['control', 'refund_rate']
treatment_refund_rate = refund_summary.loc['treatment', 'refund_rate']
refund_rate_difference = treatment_refund_rate - control_refund_rate

refund_z_statistic, refund_p_value = proportions_ztest(
    count=[treatment_refunded, control_refunded],
    nobs=[treatment_orders, control_orders],
    alternative='two-sided'
)

### 11.3 Refund Guardrail 结果

结果用于判断当前数据是否显示 Treatment 退款率显著恶化，不将未显著结果解释为证明 Treatment 不会增加退款。

In [13]:
refund_result = pd.Series({
    'Control Orders': control_orders,
    'Treatment Orders': treatment_orders,
    'Control Refunded Orders': control_refunded,
    'Treatment Refunded Orders': treatment_refunded,
    'Control Refund Rate': f'{control_refund_rate:.2%}',
    'Treatment Refund Rate': f'{treatment_refund_rate:.2%}',
    'Difference': f'{refund_rate_difference * 100:+.2f} percentage points',
    'Z statistic': f'{refund_z_statistic:.4f}',
    'p-value': f'{refund_p_value:.6f}'
})
display(refund_result.to_frame('Result'))

if refund_p_value >= 0.05:
    print('当前数据没有发现明确证据表明 Treatment 导致退款率显著恶化。')
elif refund_rate_difference > 0:
    print('当前数据表明 Treatment 的退款率显著高于 Control，需要关注退款风险。')
else:
    print('当前数据表明 Treatment 的退款率显著低于 Control。')

,Result
Control Orders,300
Treatment Orders,410
Control Refunded Orders,20
Treatment Refunded Orders,39
Control Refund Rate,6.67%
Treatment Refund Rate,9.51%
Difference,+2.85 percentage points
Z statistic,1.3568
p-value,0.174833


当前数据没有发现明确证据表明 Treatment 导致退款率显著恶化。
